ok so caietelul asta mai are greseli mai ales la partea aia cu obiecte, categorical / numerical values dar womp womp imi e ft lene sa dau fix acum la to tbleh :P

In [1]:
import pandas as pd

path = '/content/melb-housing.csv'

data = pd.read_csv(path)
data.describe()

,Rooms,Price,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,13580.000000,1.358000e+04,13580.000000,13580.000000,13580.000000,13580.000000,13518.000000,13580.000000,7130.000000,8205.000000,13580.000000,13580.000000,13580.000000
mean,2.937997,1.075684e+06,10.137776,3105.301915,2.914728,1.534242,1.610075,558.416127,151.967650,1964.684217,-37.809203,144.995216,7454.417378
std,0.955748,6.393107e+05,5.868725,90.676964,0.965921,0.691712,0.962634,3990.669241,541.014538,37.273762,0.079260,0.103916,4378.581772
min,1.000000,8.500000e+04,0.000000,3000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1196.000000,-38.182550,144.431810,249.000000
25%,2.000000,6.500000e+05,6.100000,3044.000000,2.000000,1.000000,1.000000,177.000000,93.000000,1940.000000,-37.856822,144.929600,4380.000000
50%,3.000000,9.030000e+05,9.200000,3084.000000,3.000000,1.000000,2.000000,440.000000,126.000000,1970.000000,-37.802355,145.000100,6555.000000
75%,3.000000,1.330000e+06,13.000000,3148.000000,3.000000,2.000000,2.000000,651.000000,174.000000,1999.000000,-37.756400,145.058305,10331.000000
max,10.000000,9.000000e+06,48.100000,3977.000000,20.000000,8.000000,10.000000,433014.000000,44515.000000,2018.000000,-37.408530,145.526350,21650.000000


In [5]:
y = data.Price
from sklearn.model_selection import train_test_split

X_full = data.drop('Price', axis=1)
X_train_full, X_val_full, y_train, y_val = train_test_split(X_full, y, random_state=0)

s = (X_train_full.dtypes=='object')
objcols = list(s[s].index)

X_train = X_train_full.drop(objcols, axis=1)
X_val = X_val_full.drop(objcols, axis=1)

cwmiss = [col for col in X_train.columns if X_train[col].isnull().any()]

reduced_X_train = X_train.drop(cwmiss, axis=1)
reduced_X_val = X_val.drop(cwmiss, axis=1)

In [6]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

def mae(max_leaf_nodes, X_train, X_val, y_train, y_val):
  model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=0)
  model.fit(X_train, y_train)
  preds = model.predict(X_val)
  return mean_absolute_error(y_val, preds)

def score(X_train, X_val, y_train, y_val):
  mini = 1000000000
  for max_leaf_nodes in [5, 50, 500, 5000]:
    my_mae = mae(max_leaf_nodes, X_train, X_val, y_train, y_val)
    if (my_mae < mini):
      mini = my_mae
  return mini


In [8]:
score(reduced_X_train, reduced_X_val, y_train, y_val)

224126.0461521274

In [9]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer()
imputed_X_train = pd.DataFrame(imputer.fit_transform(X_train))
imputed_X_val = pd.DataFrame(imputer.transform(X_val))

imputed_X_train.columns = X_train.columns
imputed_X_val.columns = X_val.columns

score(imputed_X_train, imputed_X_val, y_train, y_val)

226426.03307155173

In [10]:
X_train_plus = X_train.copy()
X_val_plus = X_val.copy()

for col in cwmiss:
  X_train_plus[col + '_was_missing'] = X_train_plus[col].isnull()
  X_val_plus[col + '_was_missing'] = X_val_plus[col].isnull()

imputer = SimpleImputer()
imputed_X_train_plus = pd.DataFrame(imputer.fit_transform(X_train_plus))
imputed_X_val_plus = pd.DataFrame(imputer.transform(X_val_plus))

imputed_X_train_plus.columns = X_train_plus.columns
imputed_X_val_plus.columns = X_val_plus.columns

score(imputed_X_train_plus, imputed_X_val_plus, y_train, y_val)


228669.14443620582

In [11]:
s = (X_train.dtypes=='object')
objcols = list(s[s].index)

objcols

[]

In [12]:
from sklearn.preprocessing import OrdinalEncoder

label_X_train = X_train.copy()
label_X_val = X_val.copy()

ordinal_encoder = OrdinalEncoder()
label_X_train[objcols] = ordinal_encoder.fit_transform(X_train[objcols])
label_X_val[objcols] = ordinal_encoder.transform(X_val[objcols])

score(label_X_train, label_X_val, y_train, y_val)

222530.63445744483

In [13]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder()
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[objcols]))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(X_val[objcols]))

OH_cols_train.index = X_train.index
OH_cols_valid.index = X_val.index


num_X_train = X_train.drop(objcols, axis=1)
num_X_valid = X_val.drop(objcols, axis=1)

OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)

score(OH_X_train, OH_X_valid, y_train, y_val)

222530.63445744483

In [22]:
from xgboost import XGBRegressor

model = XGBRegressor(n_estimators=1000, learning_rate=0.05, n_jobs=4)
model.fit(X_train, y_train,
            # early_stopping_rounds=5,
             eval_set=[(X_val, y_val)],
             verbose=False)

preds = model.predict(X_val)
mean_absolute_error(preds, y_val)

164131.51768225332